# Fine-tune LLaMA 3.1 8B Instruct with LoRA (QLoRA)

This notebook fine-tunes **Meta LLaMA 3.1 8B Instruct** on competitive-programming style Python solutions using **4-bit QLoRA**.

**Dataset format** (`train.jsonl`, `val.jsonl`): each line is JSON with:
- `difficulty`: e.g. introductory, interview, competition
- `question`: problem statement
- `solution`: Python solution code

**Requirements**
- NVIDIA GPU with ~16 GB+ VRAM (Colab T4 may need smaller batch size)
- Hugging Face account with access to [meta-llama/Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
- Dataset files at the paths configured below


## No local GPU? Use Google Colab (recommended)

Fine-tuning LLaMA 3.1 8B needs a **CUDA GPU**. A laptop CPU is too slow; Apple Silicon (MPS) does not support 4-bit QLoRA well.

**Recommended path:** upload this notebook to [Google Colab](https://colab.research.google.com/), enable a GPU runtime, and upload your datasets to Google Drive.

1. Upload `train.jsonl` and `val.jsonl` to Google Drive (e.g. `MyDrive/fine-tune/`)
2. In Colab: **Runtime → Change runtime type → T4 GPU**
3. Run the **Colab setup** cell below, then continue with the rest of the notebook
4. After training, download `./llama31-8b-pie-perf-lora/final_model` from Drive or Colab files

**Alternatives:** [Kaggle Notebooks](https://www.kaggle.com/code) (30 h GPU/week), [Hugging Face Jobs](https://huggingface.co/docs/autotrain/main/en/jobs), or paid GPU hosts (RunPod, Lambda Labs).

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

print("Running in Colab:" if IN_COLAB else "Running locally:", IN_COLAB)

## 1. Configuration


In [ ]:
import os
from pathlib import Path

# --- paths ---
if IN_COLAB:
    DATA_DIR = Path("/content/drive/MyDrive/fine-tune")  # put train.jsonl + val.jsonl here
    TRAIN_PATH = DATA_DIR / "train.jsonl"
    VAL_PATH = DATA_DIR / "val.jsonl"
    OUTPUT_DIR = Path("/content/drive/MyDrive/llama31-8b-pie-perf-lora")
    BATCH_SIZE = 2          # safer on Colab T4 (16 GB)
    GRAD_ACCUM = 16         # effective batch size still 32
else:
    TRAIN_PATH = Path("/Users/fabiomar/Downloads/train.jsonl")
    VAL_PATH = Path("/Users/fabiomar/Downloads/val.jsonl")
    OUTPUT_DIR = Path("./llama31-8b-pie-perf-lora")
    BATCH_SIZE = 4
    GRAD_ACCUM = 8

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# training hyperparameters
MAX_SEQ_LENGTH = 2048
MAX_STEPS = 400
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32

for p in (TRAIN_PATH, VAL_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Missing dataset: {p}\n"
            + ("Upload train.jsonl and val.jsonl to Google Drive and set DATA_DIR." if IN_COLAB else "")
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Out:   {OUTPUT_DIR.resolve()}")
print(f"Batch: {BATCH_SIZE} x grad_accum {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective")


## 2. Install dependencies (run once)


In [ ]:
# Run once in Colab or a fresh environment
!pip install -U transformers peft accelerate bitsandbytes datasets trl


## 3. Hugging Face login


In [ ]:
from huggingface_hub import login

# In Colab: run login() and paste your token, or set HF_TOKEN in Colab secrets
if IN_COLAB:
    login()
else:
    # export HF_TOKEN=... in your shell, or uncomment:
    # login()
    pass


## 4. System check


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected. QLoRA 4-bit training requires an NVIDIA GPU.")


## 5. Load model and tokenizer (4-bit QLoRA)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(model.config.model_type, model.config.max_position_embeddings)


## 6. Load datasets


In [ ]:
import json
from datasets import Dataset

def load_jsonl(path: Path) -> Dataset:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return Dataset.from_list(rows)

train_dataset = load_jsonl(TRAIN_PATH)
eval_dataset = load_jsonl(VAL_PATH)

print(train_dataset)
print(eval_dataset)
print(train_dataset[0].keys())
print("Sample difficulty:", train_dataset[0]["difficulty"])


## 7. Prompt formatting (LLaMA 3.1 chat template)


In [ ]:
SYSTEM_PROMPT = """You are an expert coding assistant.
Given a coding question, generate a complete and correct Python solution enclosed in triple backticks like:
```python
<code here>
Only return valid Python code. No explanation, no comments, no print statements, no example usage. Do not include any text outside the code block.
"""

def build_messages(example):
    user_content = f"""# difficulty: {example['difficulty']}

# question:
{example['question']}

# solution:"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["solution"]},
    ]

def formatting_prompts_func(example):
    return tokenizer.apply_chat_template(
        build_messages(example),
        tokenize=False,
        add_generation_prompt=False,
    )

# preview one training example
print(formatting_prompts_func(train_dataset[0])[:1200], "...")


## 8. LoRA setup


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 9. Train with SFTTrainer


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=50,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="none",
    group_by_length=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    formatting_func=formatting_prompts_func,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

trainer.train()


## 10. Save adapter


In [ ]:
final_dir = OUTPUT_DIR / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

print(f"Saved LoRA adapter to {final_dir.resolve()}")


## 11. Inference smoke test


In [ ]:
import torch
from peft import PeftModel

# reload base + adapter for a clean inference path
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
inference_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR / "final_model")
inference_model.eval()

sample = eval_dataset[0]
messages = build_messages(sample)[:2]  # system + user only
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)
with torch.no_grad():
    output = inference_model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("--- Generated solution ---")
print(generated[:1500])
print("\n--- Reference solution (first 500 chars) ---")
print(sample["solution"][:500])
